In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

from xgboost import XGBRegressor


# ======================
# LOAD DATA
# ======================
df = pd.read_csv("../cleaned_auction_data.csv")

# Feature engineering
df["SaleDate"] = pd.to_datetime(df["SaleDate"])
df["SaleMonth"] = df["SaleDate"].dt.month
df["SaleYear"] = df["SaleDate"].dt.year


# ======================
# TARGET
# ======================
y = df["SellingPrice"]

# Drop leakage columns
X = df.drop(columns=[
    "SellingPrice",
    "LogSellingPrice",
    "PriceVsMMR",
    "PctDiff",
    "SaleDate"
])


# ======================
# COLUMNS
# ======================
cat_cols = [
    "Make","Model","Body","Transmission","State",
    "AgeGroup","MileageGroup","MarketSignal"
]

num_cols = [
    "Year","VehicleAge","Odometer","MMR",
    "ConditionValue","SaleYear","SaleMonth"
]


# ======================
# PREPROCESSING
# ======================
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ],
    remainder="passthrough"
)


# ======================
# TRAIN TEST SPLIT
# ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# ======================
# MODEL 1: RIDGE
# ======================
ridge = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", Ridge(alpha=1.0))
])

ridge.fit(X_train, y_train)
pred_ridge = ridge.predict(X_test)

ridge_mae = mean_absolute_error(y_test, pred_ridge)
ridge_r2 = r2_score(y_test, pred_ridge)


# ======================
# MODEL 2: XGBOOST
# ======================
xgb = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ))
])

xgb.fit(X_train, y_train)
pred_xgb = xgb.predict(X_test)

xgb_mae = mean_absolute_error(y_test, pred_xgb)
xgb_r2 = r2_score(y_test, pred_xgb)


# ======================
# PRINT RESULTS
# ======================
print("RIDGE MAE:", ridge_mae)
print("RIDGE R2 :", ridge_r2)

print("XGB MAE  :", xgb_mae)
print("XGB R2   :", xgb_r2)


# ======================
# SAVE BEST MODEL
# ======================
if xgb_mae < ridge_mae:
    joblib.dump(xgb, "best_model.pkl")
    print("Saved: XGBoost")
else:
    joblib.dump(ridge, "best_model.pkl")
    print("Saved: Ridge")

RIDGE MAE: 1064.5977590852435
RIDGE R2 : 0.9527818335444782
XGB MAE  : 592.0532953070765
XGB R2   : 0.9862230857757508
Saved: XGBoost
